Modelling of df_modelling_dimensionless_pf.xlsx

# Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
import sys

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [3]:
RANDOM_STATE = 42

In [4]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [5]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

In [6]:
dataset_filename = 'df_modelling_dimensionless_pf'

# Splashing

## Train/test datasets creation

In [7]:
target = 'splashing'
train, test = get_train_test(
    dataset_filename=dataset_filename,
    target=target)

## Numerical, categorical features selection

In [8]:
train.describe()

,splashing,wettability,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,Re^(1/4),We^(1/4)^2,We^(1/4)^2 Re^(1/4)
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000
mean,0.646465,0.912458,9.242424,0.117845,0.990754,0.727273,0.025974,6.391598,26.427325,171.713124
std,0.478874,0.825530,15.987172,0.322969,0.269354,0.446113,0.024387,2.367570,7.825824,84.738011
min,0.000000,0.000000,0.000000,0.000000,0.381356,0.000000,0.011339,4.003811,11.916747,52.684981
25%,0.000000,0.000000,0.000000,0.000000,0.877193,0.000000,0.012652,4.951214,25.096339,128.988793
50%,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.013577,5.092815,28.410070,152.487586
75%,1.000000,2.000000,20.000000,0.000000,1.016949,1.000000,0.018694,8.868391,30.065485,236.927284
max,1.000000,2.000000,45.000000,1.000000,1.864407,1.000000,0.103774,11.761989,45.132225,472.780129


In [9]:
for column in train.columns:
    print(f'{column}:\t|\t{[round(x, 3) for x in train[column].unique()[:5]]}\t|\tUnique vales: {train[column].nunique()}')

splashing:	|	[0, 1]	|	Unique vales: 2
wettability:	|	[1, 2, 0]	|	Unique vales: 3
inclination:	|	[45, 0, 20]	|	Unique vales: 3
roughness_binary:	|	[0, 1]	|	Unique vales: 2
particle_liquid_density_ratio:	|	[0.847, 1.0, 1.864, 1.017, 1.22]	|	Unique vales: 8
volume_fraction_binary:	|	[1, 0]	|	Unique vales: 2
particle_droplet_diameter_ratio:	|	[0.014, 0.044, 0.014, 0.013, 0.012]	|	Unique vales: 148
Re^(1/4):	|	[4.163, 10.437, 11.345, 8.89, 5.111]	|	Unique vales: 194
We^(1/4)^2:	|	[14.229, 25.84, 37.399, 13.258, 30.323]	|	Unique vales: 197
We^(1/4)^2 Re^(1/4):	|	[59.242, 269.684, 424.301, 117.864, 154.976]	|	Unique vales: 197


In [14]:
binary_features = ['roughness_binary', 'volume_fraction_binary']
categorical_features = ['wettability']
numerical_features = ['particle_droplet_diameter_ratio',
                      'particle_liquid_density_ratio', 'Re^(1/4)', 'We^(1/4)^2', 
                      'We^(1/4)^2 Re^(1/4)', 'inclination']

In [15]:
assert len((set(train.columns)) & set(binary_features + categorical_features + numerical_features)) + 1 \
    == len(train.columns), 'Some features are not selected'

## Modelling

Models:
* Sklearn models
    * LogReg
    * SVM
    * KNN
* CatBoost
* XGBoost

### Sklearn models

In [16]:
models = [
    LogisticRegression(),
    SVC(),
    KNeighborsClassifier()]
encoders = ['onehot', 'ordenc']
smote_flags = [True, False]

for model in tqdm(models):
    for encoder in encoders:
        for smote_flag in smote_flags:
            pipeline = MLPipeline(train, test, target, model,
                    numerical_features, categorical_features,
                    random_state=RANDOM_STATE,
                    postfix=encoder,
                    smote=smote_flag,
                    dataset_filename=dataset_filename)
            pipeline.full_pipeline()

 33%|███▎      | 1/3 [00:00<00:00,  3.00it/s]

logisticregression_smote_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
logisticregression_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
logisticregression_smote_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
logisticregression_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


 67%|██████▋   | 2/3 [00:00<00:00,  3.79it/s]

svc_smote_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
svc_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
svc_smote_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
svc_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


100%|██████████| 3/3 [00:00<00:00,  3.78it/s]

kneighborsclassifier_smote_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
kneighborsclassifier_splashing_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
kneighborsclassifier_smote_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
kneighborsclassifier_splashing_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


### Boosting Models

In [17]:
models =  [
    XGBClassifier(),
    CatBoostClassifier(verbose=False)
    ]

for model in tqdm(models):
    for smote_flag in smote_flags:
        pipeline = MLPipeline(train, test, target, model,
                    numerical_features, categorical_features,
                    random_state=RANDOM_STATE,
                    postfix='', dataset_filename=dataset_filename,
                    smote=smote_flag)
        pipeline.full_pipeline()

 50%|█████     | 1/2 [00:00<00:00,  3.76it/s]

xgbclassifier_smote_splashing_df_modelling_dimensionless_pf was saved in modelling2_models
xgbclassifier_splashing_df_modelling_dimensionless_pf was saved in modelling2_models


C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future ver

catboostclassifier_smote_splashing_df_modelling_dimensionless_pf was saved in modelling2_models


C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
100%|██████████| 2/2 [00:34<00:00, 17.40s/it]

catboostclassifier_splashing_df_modelling_dimensionless_pf was saved in modelling2_models


# Net impact

## Train/test datasets creation

In [18]:
target = 'net_impact'
train, test = get_train_test(
    dataset_filename=dataset_filename,
    target=target)

## Modelling

Models:
* Sklearn models
    * LogReg
    * SVM
    * KNN
* CatBoost
* XGBoost

### Sklearn models

In [19]:
models = [
    LogisticRegression(),
    SVC(),
    KNeighborsClassifier()]
encoders = ['onehot', 'ordenc']
smote_flags = [True, False]

for model in tqdm(models):
    for encoder in encoders:
        for smote_flag in smote_flags:
            pipeline = MLPipeline(train, test, target, model,
                    numerical_features, categorical_features,
                    random_state=RANDOM_STATE,
                    postfix=encoder, dataset_filename=dataset_filename,
                    smote=smote_flag)
            pipeline.full_pipeline()

 33%|███▎      | 1/3 [00:00<00:00,  4.55it/s]

logisticregression_smote_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
logisticregression_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
logisticregression_smote_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
logisticregression_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


 67%|██████▋   | 2/3 [00:00<00:00,  4.54it/s]

svc_smote_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
svc_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
svc_smote_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
svc_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


100%|██████████| 3/3 [00:00<00:00,  4.23it/s]

kneighborsclassifier_smote_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
kneighborsclassifier_net_impact_df_modelling_dimensionless_pf_onehot was saved in modelling2_models
kneighborsclassifier_smote_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models
kneighborsclassifier_net_impact_df_modelling_dimensionless_pf_ordenc was saved in modelling2_models


### Boosting Models

In [20]:
models =  [
    XGBClassifier(),
    CatBoostClassifier(verbose=False)]

for model in tqdm(models):
    for smote_flag in smote_flags:
        pipeline = MLPipeline(train, test, target, model,
                    numerical_features, categorical_features,
                    random_state=RANDOM_STATE,
                    postfix='', dataset_filename=dataset_filename,
                    smote=smote_flag)
        pipeline.full_pipeline()

 50%|█████     | 1/2 [00:00<00:00,  3.20it/s]

xgbclassifier_smote_net_impact_df_modelling_dimensionless_pf was saved in modelling2_models
xgbclassifier_net_impact_df_modelling_dimensionless_pf was saved in modelling2_models


C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future ver

catboostclassifier_smote_net_impact_df_modelling_dimensionless_pf was saved in modelling2_models


C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
100%|██████████| 2/2 [00:34<00:00, 17.04s/it]

catboostclassifier_net_impact_df_modelling_dimensionless_pf was saved in modelling2_models
